In [16]:
import cv2
from ultralytics import YOLO
import pandas as pd
import numpy as np
from hmmlearn import hmm
import glob
import os

In [17]:
model = YOLO('../best_yolov8_coral_reef/runs/detect/reef_coral/weights/best.pt')
video_path = '../yolov8_model/videos/Qualification 72 - 2025 Iowa Regional.mp4'
cap = cv2.VideoCapture(video_path)

In [18]:
# Dictionary to store logs for multiple reefs
reef_logs = {} 
last_known_reefs = {} 
FACE_CAPACITIES = [3, 1, 2, 2, 1, 3]

fps = cap.get(cv2.CAP_PROP_FPS)
stop_frame = int(2.8 * 60 * fps)
frame_count = 0

print("Processing video...")

while cap.isOpened():
    if frame_count > stop_frame: break
    success, frame = cap.read()
    if not success: break

    h, w, _ = frame.shape
    y_offset = int(h * (3/5))
    cropped_frame = frame[y_offset:h, 0:w]
    
    # RUN TRACKER: persist=True is key for keeping IDs across frames
    results = model.track(cropped_frame, persist=True, conf=0.3, verbose=False)
    
    frame_detections = []

    # 1. Parse all detections in the current frame
    if results[0].boxes is not None and results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.tolist()
        ids = results[0].boxes.id.int().tolist()
        clss = results[0].boxes.cls.int().tolist()

        for box, obj_id, cls in zip(boxes, ids, clss):
            label = model.names[cls]
            if label == 'reef':
                last_known_reefs[obj_id] = box
            else:
                # Store corals and robots to check against all reefs
                frame_detections.append({'label': label, 'coords': box})

    # 2. Score every reef we have ever seen
    # This ensures even if a reef is momentarily hidden, we log a state for it
    for r_id, r_box in last_known_reefs.items():
        if r_id not in reef_logs:
            reef_logs[r_id] = []
            print(f"New Reef ID detected: {r_id}")

        rx1, ry1, rx2, ry2 = r_box
        reef_w, reef_h = rx2 - rx1, ry2 - ry1
        l4_top, l4_bottom = ry1 - (reef_h * 0.05), ry1 + (reef_h * 0.30)
        face_width = reef_w / 6
        
        frame_obs = {"frame": frame_count}
        
        for face_idx in range(6):
            fx1 = rx1 + (face_idx * face_width)
            fx2 = fx1 + face_width
            count = 0
            for det in frame_detections:
                dx1, dy1, dx2, dy2 = det['coords']
                mx, my = (dx1 + dx2) / 2, (dy1 + dy2) / 2
                if fx1 <= mx <= fx2 and l4_top <= my <= l4_bottom:
                    if det['label'] == 'coral': 
                        count += 1
            
            frame_obs[f"Face_{face_idx}_Count"] = min(count, FACE_CAPACITIES[face_idx])
        
        reef_logs[r_id].append(frame_obs)

    if frame_count % 500 == 0:
        print(f"Frame {frame_count} - Tracking {len(last_known_reefs)} reef(s)...")

    frame_count += 1

cap.release()

# 3. Save files
if not reef_logs:
    print("Error: No reefs were tracked. Try lowering conf to 0.3.")
else:
    for r_id, logs in reef_logs.items():
        filename = f'reef_{r_id}_observations.csv'
        pd.DataFrame(logs).to_csv(filename, index=False)
        print(f"Saved {filename}")

Processing video...
Frame 0 - Tracking 0 reef(s)...
New Reef ID detected: 14
New Reef ID detected: 27
New Reef ID detected: 40
New Reef ID detected: 41
Frame 500 - Tracking 4 reef(s)...
Frame 1000 - Tracking 4 reef(s)...
Frame 1500 - Tracking 4 reef(s)...
Frame 2000 - Tracking 4 reef(s)...
Frame 2500 - Tracking 4 reef(s)...
Frame 3000 - Tracking 4 reef(s)...
Frame 3500 - Tracking 4 reef(s)...
Frame 4000 - Tracking 4 reef(s)...
New Reef ID detected: 932
New Reef ID detected: 935
New Reef ID detected: 970
Frame 4500 - Tracking 7 reef(s)...
Frame 5000 - Tracking 7 reef(s)...
Saved reef_14_observations.csv
Saved reef_27_observations.csv
Saved reef_40_observations.csv
Saved reef_41_observations.csv
Saved reef_932_observations.csv
Saved reef_935_observations.csv
Saved reef_970_observations.csv


In [19]:
# Define capacities for the 6 faces (consistent with your previous logic)
CAPACITIES = [3, 1, 2, 2, 1, 3]

def build_face_hmm(capacity):
    num_states = capacity + 1
    # n_trials=3 because that is the max possible observation (3 corals)
    model = hmm.MultinomialHMM(n_components=num_states, n_trials=3, n_iter=100)
    
    # 1. Start Probabilities: Assume match starts with 0 corals
    model.startprob_ = np.zeros(num_states)
    model.startprob_[0] = 1.0
    
    # 2. Transition Matrix (A): Probability of moving from state i to state j
    trans_mtx = np.zeros((num_states, num_states))
    for i in range(num_states):
        for j in range(num_states):
            if i == j:
                trans_mtx[i][j] = 0.90  # Stay in current state
            elif j == i + 1:
                trans_mtx[i][j] = 0.09  # Score a coral
            elif j < i:
                trans_mtx[i][j] = 0.01  # Rare: Coral falls off or miscount
        trans_mtx[i] /= trans_mtx[i].sum()
    model.transmat_ = trans_mtx
    
    # 3. Emission Matrix (B): Prob of seeing 'obs' given true state 'i'
    # Shape must be (num_states, 4) for observations 0, 1, 2, 3
    emit_mtx = np.full((num_states, 4), 0.05)
    for i in range(num_states):
        emit_mtx[i][min(i, 3)] = 0.85 # Expect to see 'i' corals
    
    for i in range(num_states):
        emit_mtx[i] /= emit_mtx[i].sum()
        
    model.emissionprob_ = emit_mtx
    return model

# Find all reef observation files
csv_files = glob.glob('reef_*_observations.csv')

if not csv_files:
    print("No observation files found. Did the tracking script run successfully?")
else:
    print(f"Found {len(csv_files)} reef(s). Processing...")

    for file_path in csv_files:
        reef_id = file_path.split('_')[1] # Extract ID from filename
        df = pd.read_csv(file_path)
        
        reef_final_score = 0
        
        # Process each of the 6 faces
        for i, cap in enumerate(CAPACITIES):
            model_hmm = build_face_hmm(cap)
            
            # 1. Get raw observations
            raw_obs = df[f"Face_{i}_Count"].values
            
            # 2. One-hot encode for MultinomialHMM
            obs_one_hot = np.zeros((len(raw_obs), 4), dtype=int)
            for row_idx, val in enumerate(raw_obs):
                obs_one_hot[row_idx, min(int(val), 3)] = 1
            
            # 3. Decode the most likely sequence
            try:
                logprob, state_sequence = model_hmm.decode(obs_one_hot, algorithm="viterbi")
                df[f"Face_{i}_HMM_State"] = state_sequence
                reef_final_score += state_sequence[-1]
            except Exception as e:
                print(f"Error decoding Face {i} for Reef {reef_id}: {e}")

        # Save the smoothed results
        output_name = f'reef_{reef_id}_final_results.csv'
        df.to_csv(output_name, index=False)
        
        print(f"--- REEF {reef_id} ---")
        print(f"Final L4 Score: {reef_final_score}")
        print(f"Results saved to {output_name}\n")

print("All reefs processed.")

MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows the standard definition for a Multinomial distribution (e.g. as in https://en.wikipedia.org/wiki/Multinomial_distribution). See these issues for details:
https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340
MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows the standard definition for a Multinomial distribution (e.g. as in https://en.wikipedia.org/wiki/Multinomial_distribution). See these issues for details:
https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340
MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows

Found 7 reef(s). Processing...
--- REEF 935 ---
Final L4 Score: 0
Results saved to reef_935_final_results.csv

--- REEF 932 ---
Final L4 Score: 1
Results saved to reef_932_final_results.csv

--- REEF 14 ---
Final L4 Score: 0
Results saved to reef_14_final_results.csv

--- REEF 27 ---
Final L4 Score: 0
Results saved to reef_27_final_results.csv



MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows the standard definition for a Multinomial distribution (e.g. as in https://en.wikipedia.org/wiki/Multinomial_distribution). See these issues for details:
https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340
MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows the standard definition for a Multinomial distribution (e.g. as in https://en.wikipedia.org/wiki/Multinomial_distribution). See these issues for details:
https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340
MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows

--- REEF 41 ---
Final L4 Score: 0
Results saved to reef_41_final_results.csv

--- REEF 970 ---
Final L4 Score: 5
Results saved to reef_970_final_results.csv

--- REEF 40 ---
Final L4 Score: 1
Results saved to reef_40_final_results.csv

All reefs processed.
